# Train, review, test and utilize machine learning algorithm dAeDFM (nonlinear Dynamic Factor Model by Deep Autoencoder) from AWS Marketplace

--- ---

## Overview of the algorithm

Our dAeDFM algorithm is a nonlinear dynamic factor model (DFM) formulated as deep autoencoder (DAE) with dense/causal nowcasting/forecasting encoding/decoding sub-layers and with robust squared-L1 loss criterion.

By nonlinearization of well-established dynamic factor models (DFM) on large sets of multiple time-series, we introduce nonlinear DFM formulated as deep autoencoders (DAE) composed of sub-layers of dense nowcasting, dense forecasting, causal nowcasting and causal forecasting.

Different DAE configurations mixing the dense/causal nowcasting/forecasting layers can make the nonlinear DAE model serving differently as total nowcasting, predictive nowcasting, one-step forecasting, enhanced forecasting and total forecasting models.

To resist outliers in input data (other than those in output-target difference), we introduce robust squared-L1 loss criterion. Our squared-L1 loss function having both outlier-resistant robustness of L1 loss (mean absolute error) and continuous differentiability of L2 loss (mean squared error).

In addition, separate validation and testing datasets can be formed for nonlinear model training and evaluation using our observation-based input data sample variation approach.

WHY deep autoencoder?
1. Neural networks introduce nonlinearity by applying certain element-wise nonlinear activation functions to linear combinations (weighted sums).
1. Multi-layered (deep) neural nets add more nonlinearity by nested (element-wise) nonlinear functions.
1. Neural networks with latent layer of hidden neurons are validated by proven and widely accepted Universal Approximation Theorem.
1. Autoencoders (a specific category of neural nets) were initially formulated as nonlinear extensions of linear principal components analysis (PCA).
1. Both time-domain PCA and frequency-domain (spectral) PCA are among core methods to formulate linear DFM models (Ling, 2026).
1. Huge successes of large language models (LLM) which are deep neural networks and are underlying the current AI boom worldwide

WHAT steps of nonlinearization to formulate DFM as DAE?
1. Classic linear dynamic factor model.
1. Nonlinear DFM by nonlinear wrapper.
1. Autoencoder with causal nowcasting layer.
1. Causal nowcasting sub-layers.
1. Dense nowcasting sub-layers.
1. Dense forecasting sub-layers.
1. Causal forecasting sub-layers.
1. Deep autoencoder (DAE, multiple layers).

This sample notebook shows you how to train, review and deploy the [Nonlinear Dynamic Factor Model by Deep Autoencoder (dAeDFM) Algorithm](https://aws.amazon.com/marketplace/pp) from AWS Marketplace.

**Note**: This is a reference notebook and it cannot run unless you make changes suggested in the notebook.

## Pre-requisites

1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that IAM role used has **AmazonSageMakerFullAccess**
1. Some hands-on experience using [Amazon SageMaker](https://aws.amazon.com/sagemaker/).
1. To use this algorithm successfully, ensure that:
    1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used: 
        1. **aws-marketplace:ViewSubscriptions**.
        1. **aws-marketplace:Unsubscribe**.
        1. **aws-marketplace:Subscribe**.
    2. or your AWS account has a subscription to [Nonlinear Dynamic Factor Model by Deep Autoencoder (dAeDFM) Algorithm](https://aws.amazon.com/marketplace/pp). 

## Contents

1. [Subscribe to the algorithm](#1.-Subscribe-to-the-algorithm)
    1. [Subscription](#1.1.-Subscription)
    1. [Prepare relevant environment](#1.2.-Prepare-relevant-environment)
1. [Prepare dataset](#2.-Prepare-dataset)
	1. [Dataset format expected by the algorithm](#2.1.-Dataset-format-expected-by-the-algorithm)
	1. [Configure and visualize train and test dataset](#2.2.-Configure-and-visualize-train-and-test-dataset)
	1. [Upload datasets to Amazon S3](#2.3.-Upload-datasets-to-Amazon-S3)
1. [Train a machine learning model](#3.-Train-a-machine-learning-model)
	1. [Set up environment](#3.1.-Set-up-environment)
    1. [Set hyperparameters](#3.2.-Set-hyperparameters)
	1. [Train a model](#3.3.-Train-a-model)
1. [Review and test trained model](#4.-Review-and-test-trained-model)
    1. [Trained model](#4.1.-Trained-model)
    1. [Create input payload](#4.2.-Create-input-payload)
    1. [Generate trained model report](#4.3.-Generate-trained-model-report)
    1. [Review trained model report](#4.4.-Review-trained-model-report)
    1. [Make rolling model test](#4.5.-Make-rolling-model-test)
    1. [Review rolling model test](#4.6.-Review-rolling-model-test)
1. [Perform Batch inference](#5.-Perform-Batch-inference)
1. [Clean-up](#6.-Clean-up)
	1. [Delete the model](#6.1.-Delete-the-model)
	1. [Unsubscribe to the listing (optional)](#6.2.-Unsubscribe-to-the-listing-(optional))

## Usage instructions

You can run this notebook one cell at a time (By using Shift+Enter for running a cell).

## 1. Subscribe to the algorithm

### 1.1. Subscription

To subscribe to the algorithm:
1. Open the algorithm listing page [Nonlinear Dynamic Factor Model by Deep Autoencoder (dAeDFM) Algorithm](https://aws.amazon.com/marketplace/pp).
1. On the AWS Marketplace listing,  click on **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you agree with EULA, pricing, and support terms. 
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn**. This is the algorithm ARN that you need to specify while training a custom ML model. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
# Specify algorithm ARN (of the algorithm subscribed from AWS Marketplace)
# my_algorithm_name = 'arn:aws:sagemaker:{region}:123456789012:algorithm/{daedfm_algorithm_label}'
my_algorithm_name = 'arn:aws:sagemaker:us-east-1:123456789012:algorithm/dAeDFM-0'

# Specify algorithm Name (of algorithm created in your own AWS account)
my_algorithm_name = 'dAeDFM-0'

### 1.2. Prepare relevant environment

<font color='red'>**REQUIRED: Amazon AWS Sagemaker Python SDK V3 (Verion THREE).**<font>

In [ ]:
# to install Amazon AWS Sagemaker Python SDK V3, change False to True (on the following line), and run this cell
if False:
    pip install --upgrade --no-cache-dir sagemaker

In [ ]:
# sagemaker python sdk V3 for setup
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.helper.session_helper import get_execution_role

# sagemaker python sdk V3 for train, review, test and RUN dAeDFM algorithm/model
from sagemaker.core.training.configs import Compute
from sagemaker.core.training.configs import InputData
from sagemaker.core.training.configs import OutputDataConfig
from sagemaker.train.model_trainer import ModelTrainer

# s3 client / resource of boto3
import boto3
my_s3_client = boto3.client('s3')
my_s3_resource = boto3.resource('s3')

In [ ]:
# sagemaker session, bucket and role parameters to be used throughout this notebook
my_session = Session()
my_bucket = my_session.default_bucket()
my_role = get_execution_role()

# review
print('Default S3 bucket: my_bucket = {}'.format(my_bucket))
print('Execution role: my_role = {}'.format(my_role))

To run this Sagemaker notebook example, following S3 folders are expected to be in place (here my_prefix = 'daedfm'):
1. **S3://{my_bucket}/{my_prefix}/input/data/train/**
1. **S3://{my_bucket}/{my_prefix}/input/data/model/**
1. **S3://{my_bucket}/{my_prefix}/input/data/report/**
1. **S3://{my_bucket}/{my_prefix}/model/**
1. **S3://{my_bucket}/{my_prefix}/output/data/**

In [ ]:
# Specify s3 prefix for the algorithm
my_prefix = 'daedfm'

# the algorithm Docker container input data channels
input_train_channel = 'train'
input_model_channel = 'model'
input_report_channel = 'report'

# all AWS s3 paths
my_s3_path = 's3://{}/{}'.format(my_bucket, my_prefix)
my_input_data_train_path = 's3://{}/{}/input/data/train'.format(my_bucket, my_prefix)
my_input_data_model_path = 's3://{}/{}/input/data/model'.format(my_bucket, my_prefix)
my_input_data_report_path = 's3://{}/{}/input/data/report'.format(my_bucket, my_prefix)
my_model_path = 's3://{}/{}/model'.format(my_bucket, my_prefix)
my_output_data_path = 's3://{}/{}/output/data'.format(my_bucket, my_prefix)

## 2. Prepare dataset

### 2.1. Dataset format expected by the algorithm

The dAeDFM (nonlinear dynamic factor model by deep autoencoder) algorithm takes, as input data, multiple time-series data contained in a CSV (comma separated value) data table, in a format of a CSV text-string or a CSV text-file.

Structur of the CSV data table (of input data):
1. Each row of the data table is for values of an individual time-series (TS).
    1. Row header is the label or symbol of the time-series.
1. Each column is for values of all time-series at a specific moment in time.
    1. Column header is the time-index or time-stamp of the moment.
    1. The first data column is for the earliest time.
    2. The last column for the most recent time.
1. Therefore,
    1. The first row of the CSV data table is "Label/Symbol/Description, earliest time-stamp, next time-stamp, ..., most recent time-stamp".
    1. The first column of the CSV table is "Label/Symbol/Description, label of 1st TS, label of 2nd TS, ..., label of last TS".
1. The current version of dAeDFM algorithm requires equally spaced time-stamps.

The input data is essentially in the form of "Row Time-Series of Column Vector".

One of the simplest methods to generate such a CSV text-file is to save a Microsoft Excel spreadsheet as (into) a CSV file.

You can also find more information about dataset format in **Usage Information** section of [Nonlinear Dynamic Factor Model by Deep Autoencoder (dAeDFM) Algorithm](https://aws.amazon.com/marketplace/pp).

### 2.2. Configure and visualize train and test dataset

A [sample dataset](https://github.com/i4cast/aws/blob/main/nonlinear_DFM_by_deep_autoencoder/input/Weekly_VTS_6Yr.csv) that can be utilized to test, review and run the dAeDFM (nonlinear DFM by deep autoencoder) algorithm/model is six-year weekly (logarithmic) performances of mutual funds traded in the U.S. invested in equities, fixed income, and commodities. Each row of the data table is of an individual mutual fund. Each column is of a specific calendar week in history. The last week (the last column) was the week with a time-stamp as "2026-09-25" (Friday).

Following simple steps can be used to download this sample dataset to your local drive:
1. Open the webpage of this [sample dataset](https://github.com/i4cast/aws/blob/main/nonlinear_DFM_by_deep_autoencoder/input/Weekly_VTS_6Yr.csv).
1. Click [Raw] option located at top right of the data table.
1. In the Raw data window, right click [Save as].
1. Set local file folder and file name in the "Save As" window, then click [Save].

To review this sample dataset, You can open the CSV data file downloaded to your local drive by Microsoft Excel or any alternative App.

### 2.3. Upload datasets to Amazon S3

After downloading the [sample dataset](https://github.com/i4cast/aws/blob/main/nonlinear_DFM_by_deep_autoencoder/input/Weekly_VTS_6Yr.csv), you can upload the data to your AWS S3 locations:

1. {my_bucket}/{my_prefix}/input/data/train/ (for model training)
1. {my_bucket}/{my_prefix}/input/data/report/ (for trained model reporting to review or test the model)

Following simple steps can be used to upload the sample dataset to your AWS S3 locations:
1. Open AWS S3 Console.
1. First, go to S3 folder: {my_bucket}/{my_prefix}/input/data/train/.
1. Upload the saved local CSV data file to your AWS S3 folder.
1. Then, go to S3 folder: {my_bucket}/{my_prefix}/input/data/report/.
1. Upload the saved local CSV data file to your AWS S3 folder.

## 3: Train a machine learning model

Now that dataset is available in an accessible Amazon S3 bucket, we are ready to train a machine learning model. 

### 3.1. Set up environment

Define AWS compute object

In [ ]:
# AWS compute with instance type of 'ml.m5.xlarge'
my_compute = Compute(
    instance_type='ml.m5.xlarge',
    instance_count=1)

Specify input data and output path

In [ ]:
# input data for model training
my_model_training_input_data = InputData(
    channel_name=input_train_channel,
    data_source=my_input_data_train_path)

# model training output path
my_model_training_output_path = OutputDataConfig(
    s3_output_path=my_model_path)

### 3.2. Set hyperparameters

Set values of hyperparameters for dAeDFM model training.
  
Following are descriptions and sample values (for **weekly** time-series data) of hyperparameters one by one. You can also find more information in **Hyperparameters** section of [Nonlinear Dynamic Factor Model by Deep Autoencoder (dAeDFM) Algorithm](https://aws.amazon.com/marketplace/pp).

***Hyperparameters defining dynamic factor model***

In [ ]:
# len_learn_window : int, >> 1
#     Length of trailing time window of data points
#         for model learning

# example
len_learn_window = 52

In [ ]:
# var_order : int, >= 1
#     Vector autoregressive (VAR) order, p, of dynamic
#         factor model of vector time-series

# example
var_order = 13

In [ ]:
# max_forecast_step : int >= 1
#     Maximum number of forecasting steps of time-series
#         forecasts

# example
max_forecast_step = 13

In [ ]:
# target_type : str, (in target_type_list)
#     Specified type of vector, or multiple, time-series
#         data on which model inferences, predictions or
#         forecasts are made, and are evaluated
#     Type 's', 'S', 'standardized', 'b', 'B', 'base-case':
#         Inferences/forecasts are made/evaluated on
#         standardized, or base-case, values of observed
#         vector time-series
#     Type 'z', 'Z', 'zero-mean':
#         Inferences/forecasts are made/evaluated on zero-
#         mean, or de-meaned, values of observed vector
#         time-series
#     Type 'o', 'O', 'original':
#         Inferences/forecasts are made/evaluated on values
#         of original/observed vector time-series

# example
target_type = 'Original'

In [ ]:
# fwd_cumsum : bool (True, or False)
#     Whether or not to make cum-sum forecast

# example
fwd_cumsum = True

***Hyperparameters defining structure of deep autoencoder***

In [ ]:
# num_ts_dict : dict, keys (dae_layer_list)
#         obj[key] : int
#     Number of (observed or letant) time-seres
#         at specified layers in deep autoencoder
#     Notes: num_ts_input = num_ts_dict[0]
#         > num_ts_dict[1] > num_ts_dict[2] > ...
#         > num_ts_dict[num_latent_layer]

# example
# num_ts_input = 87
# num_ts = 10 (hidden code time-series)
num_ts_dict = dict({
    '0': '87',
    '1': '40',
    '2': '20',
    '3': '10'})

In [ ]:
# dae_layer_spec : dict, keys (
#             encoding_layer_list + decoding_layer_list)
#         obj[key] : str, in ['nowcast',
#             'n', 'N', 'forecast', 'f', F']
#     Nowcasting/forecasting specification
#         of each of all encoding and decoding
#         layers of deep autoencoder
#     Notes on models
#         (1) Total nowcasting
#             all encoding/decoding layers 'N'
#         (2) Predictive nowcasting
#             first encoding layer 'N', some other 'F'
#         (3) One-step forecasting
#             first encoding layer 'F', all others 'N'
#         (4) Enhanced forecasting
#             first encoding layer 'F', others 'F'/'N'
#         (5) Total forecasting
#             all encoding/decoding layers 'F'

# total nowcasting model
if False:
    dae_layer_spec = dict({
        '(87,40)': 'N', '(40,20)': 'N', '(20,10)': 'N',
        '(10,20)': 'N', '(20,40)': 'N', '(40,87)': 'N'})

# predictive nowcasting model
if False:
    dae_layer_spec = dict({
        '(87,40)': 'N', '(40,20)': 'N', '(20,10)': 'F',
        '(10,20)': 'F', '(20,40)': 'F', '(40,87)': 'N'})

# one-step forecasting model
if False:
    dae_layer_spec = dict({
        '(87,40)': 'F', '(40,20)': 'N', '(20,10)': 'N',
        '(10,20)': 'N', '(20,40)': 'N', '(40,87)': 'N'})

# enhanced forecasting model
if True:
    dae_layer_spec = dict({
        '(87,40)': 'F', '(40,20)': 'N', '(20,10)': 'N',
        '(10,20)': 'F', '(20,40)': 'F', '(40,87)': 'N'})

# total forecasting model
if False:
    dae_layer_spec = dict({
        '(87,40)': 'F', '(40,20)': 'F', '(20,10)': 'F',
        '(10,20)': 'F', '(20,40)': 'F', '(40,87)': 'F'})

In [ ]:
# causal_layer_ratio : float, 0 <= ratio <= 1
#     Weight of causal (nowcasting/forecasting) sub-
#         layer output in total nonlinear layer output
#         (from all causal and dense sub-layers)
#     Notes: bottlelneck encoding and output
#         decoding layer, always dense linear
#     Notes: this ratio to be treated
#         as 0 if < 0.05, and as 1 if > 0.95

# example
causal_layer_ratio = 0.8

In [ ]:
# init_pcwt_ratio : float, 0 <= ratio <= 1
#     Portion of PCA-based initial weight values
#         in total initial random weight values
#         of linear layers of deep autoencoder

# example
init_pcwt_ratio = 0.8

***Hyperparameters defining autoencoder training process***

In [ ]:
# num_epoch : int
#     Number of iterations taking entire input
#         datasets to train deep autoencoder

# example
num_epoch = 1000

In [ ]:
# patience : int > 1
#     Number of training epochs to run without
#         validation metric improvement before
#         early stopping

# example
patience = 10

In [ ]:
# batch_size : int, < len_learn_window
#     Size of mini-batches for mini-batch
#         gradient descent

# example
batch_size = 0

In [ ]:
# train_criterion : dict, keys ('grad', 'stats')
#         obj['grad'] : str, in [
#             'SquaredL1Loss', 'SmoothL1Loss',
#             'HuberLoss', 'MSELoss']
#         obj['stats'] : str,
#             in ['L1Loss', 'MSELoss']
#     Loss function for backprop gradient descent
#         ('grad') and for performance statistics
#         ('stats', for validation, testing and
#         reporting) in deep autoencoder model
#         training process

# example
train_criterion = dict({
    'grad': 'SquaredL1Loss',
    'stats': 'L1Loss'})

In [ ]:
# learning_rate : float, > 0 and << 1
#     Step size of gradient descent

# example
learning_rate = 0.02

In [ ]:
# input_variation_rate : dict, keys (
#             'Validation', 'Testing'),
#         obj[key] : float, 0 < rate <= 1
#     Input data sample variation rate in setting
#         up validation and testing datasets for
#         nonlinear model training and evaluation

# example
input_variation_rate = dict({
    'Validation': '0.2',
    'Testing': '0.25'})

In [ ]:
# dae_state_file : str
#     Name of .pt or .pth data file containing
#         state_dict dataset of deep autoencoder

# example
dae_state_file = 'dae_state_dict.pt'

***Hyperparameters defining trained model reporting***

In [ ]:
# model_report_list : list
#     List of names of possible trained model reports

# available model reports
model_report_list = list(dict({
     1: 'mean',
     2: 'stdev',
     3: 'vts_basecase',
     4: 'dae_data_tensor',
     5: 'dae_latent_code',
     6: 'dae_training_metric',
     7: 'basecase_train_metric',
     8: 'vts',
     9: 'dae_forecast_tensor',
    10: 'basecase_forecast',
    11: 'forecast',
    12: 'rolling_test'
}).values())

**Notes**: All individual trained model reports will be described in details at Section 4.3. "Generate trained model report" of this sample Notebook.

In [ ]:
# model_report_type : str, in ['text/csv', 'application/json']
#     Type of specific format of trained model reports

# example
model_report_type = str('text/csv')

In [ ]:
# num_forecasts : int >> 1
#     Number of rolling model trainings, in dAeDFM model test,
#     for rolling model forecasts performance evaluatoin

# example
num_forecasts = 13  # better to be 36 or more

### 3.3. Train a model

Specify a **Complete** set of model training hyperparameters

In [ ]:
# my_hyperparameters
# all individual elements must be individual strings
#
my_hyperparameters = dict({
    #
    # DFM parameters
    'len_learn_window': '52',
    'var_order': '13',
    'max_forecast_step': '13',
    'target_type': 'Original',
    'fwd_cumsum': 'True',
    #
    # DAE parameters
    'num_ts_dict': "{}".format(
        num_ts_dict),
    'dae_layer_spec': "{}".format(
        dae_layer_spec),
    'causal_layer_ratio': '0.8',
    'init_pcwt_ratio': '0.8',
    #
    # DAE training parameters
    'num_epoch': '1000',
    'patience': '10',
    'batch_size': '0',
    'train_criterion': "{}".format(
        train_criterion),
    'learning_rate': '0.02',
    'input_variation_rate': "{}".format(
        input_variation_rate),
    'dae_state_file': 'dae_state_dict.pt',
    #
    # model reporting/testing parameters
    'model_report_list': "{}".format(['forecast']),
    'model_report_type': 'text/csv',
    'num_forecasts': '13'
})

Create model training job

In [ ]:
# model trainer for model training
my_model_trainer = ModelTrainer(
    # training_mode=my_mode,
    sagemaker_session = my_session,
    role=my_role,
    # base_job_name,
    # source_code,
    # distributed,
    compute=my_compute,
    # networking,
    # stopping_condition,
    # training_image,
    # training_image_config,
    algorithm_name=my_algorithm_name,
    output_data_config=my_model_training_output_path,
    input_data_config=[  # MUST be a LIST of input data objects
        my_model_training_input_data],
    # checkpoint_config,
    training_input_mode='File',
    # environment,
    hyperparameters=my_hyperparameters
    # tags,
    # local_container_root,
    # model_package_config,
    # config_mgr
)

Run model training job

**Notes**: You can view model training logs live in CloudWatch (Console) with following steps:
1. Click / Open [CloudWatch] under (AWS) "Console Home".
1. Click [Log Management] under "Logs".
1. Check/Select [/aws/sagemaker/TrainingJobs] under "Log group".
1. Click "Start tailing" to Live Tail selected Log Group(s).

In [ ]:
# prompt
print()
print('Model training\n')
print('WAIT until training job completes and shows trained model artifact data files')

# ModelTrainer.train(wait=True, logs=True)
my_model_trainer.train(wait=True, logs=True)

# training job name and model artifact data file
# S3://{my_bucket}/{my_prefix}/model/{training_job_name}/output/model.tar.gz
training_job_name = my_model_trainer._latest_training_job.training_job_name
s3_model_uri = my_model_trainer._latest_training_job.model_artifacts.s3_model_artifacts
s3_bucket = my_s3_resource.Bucket(my_bucket)
training_output_folder = '{}/model/{}/output/'.format(my_prefix, training_job_name)

# show training job name and model artifact data file
print('Training job name:\n{}\n'.format(training_job_name))
print('Trained model artifact data file URI:\n{}\n'.format(s3_model_uri))
print('Model training output data file(s):')

# show model training output data files
for obj in s3_bucket.objects.filter(Prefix=training_output_folder):
    print(obj.key)
print()

You can review model training logs in AWS CloudWatch by following steps:
1. Click / Open [CloudWatch] under (AWS) "Console Home".
1. Click [Log Management] under "Logs".
1. Click [/aws/sagemaker/TrainingJobs] under "Log group".
1. Sort "Last event time" form latest to earliest.
1. Click / Open the first item under "Log stream".
1. Review the log lines of this model training job

## 4. Review and test trained model

After model training process, you can review and and test the trained model with a "trained model reporting" procedure

### 4.1. Trained model

To make the algorithm serving as a trained model, copy the trained model artifact data file, s3://{my_bucket}/{my_prefix}/model/{training-job-name}/output/model.tar.gz, to s3 **input data (trained) model channel**: s3://{my_bucket}/{my_prefix}/input/data/model/

In [ ]:
# source file of trained model artifact data
source_file = {
    'Bucket': my_bucket,
    'Key': '{}/model/{}/output/model.tar.gz'.format(my_prefix, training_job_name)
}

# destination file of trained model artifact data
dest_bucket = my_bucket
dest_key = '{}/input/data/model/model.tar.gz'.format(my_prefix)

# copy trained model artifact data file
my_s3_client.copy(
    source_file,
    dest_bucket,
    dest_key
)

# input model channel : s3://{my_bucket}/{my_prefix}/input/data/model
s3_bucket = my_s3_resource.Bucket(my_bucket)
input_model_folder = '{}/input/data/model/'.format(my_prefix)

# show input model files
print()
print('Trained model artifact data file(s) for model reporting and testing:')
for obj in s3_bucket.objects.filter(Prefix=input_model_folder):
    print(obj.key)
print()

### 4.2. Create input payload

Specify trained model, input data, and output path for trained model reporting

In [ ]:
# input model artifact data for trained model reporting
my_trained_model_artifact = InputData(
    channel_name=input_model_channel,
    data_source=my_input_data_model_path)

# input data for trained model reportinkg
my_model_reporting_input_data = InputData(
    channel_name=input_report_channel,
    data_source=my_input_data_report_path)

# trained model reporting output path
my_model_reporting_output_path = OutputDataConfig(
    s3_output_path=my_output_data_path,
    compression_type='NONE')

### 4.3. Generate trained model report

Available trained dAeDFM model reporting output data items

**"mean"**  

Data access method and data item(s):  

    mean_vec = dAeDFM_obj.get_mean()

    mean_vec : pandas.Series, index (ts_list)
        Sample mean vector of observed vector time-series
            of data points in last learning window

**"stdev"**  

Data access method and data item(s):  

    stdev_vec = dAeDFM_obj.get_stdev()

    stdev_vec : pandas.Series, index (ts_list)
        Sample standard deviation vector of observed
            vector time-series of data points in last
            learning window

**"vts_basecase"**  

Data access method and data item(s):  

    (vts_basecase_input, vts_basecase_output,
        vts_basecase_error
    ) = dAeDFM_obj.get_vts_basecase()

    vts_basecase_input : pd.DataFrame,
            index (ts_list), columns (asof_list)
        Standardized, base-case, time-series of observed
            vector time-series for dAeDFM model learning
            and inference

    vts_basecase_output : pd.DataFrame,
            index (ts_list), columns (asof_list)
        Output from deep autoencoder serving as modeled
            components of standardized, base-case, time-
            series of observed vector time-series

    vts_basecase_error : pd.DataFrame,
            index (ts_list), columns (asof_list)
        Part of data values not modeled by deep auto-
            encoder, or residual component of base-case
            time-series of observed vector time-series

**"dae_data_tensor"**  

Data access method and data item(s):  

    dae_input_tensor = (
        dAeDFM_obj.get_dae_data_tensor(
        dae_data_label='input'))

    dae_output_tensor = (
        dAeDFM_obj.get_dae_data_tensor(
        dae_data_label='output'))

    dae_error_tensor = (
        dAeDFM_obj.get_dae_data_tensor(
        dae_data_label='error'))

    dae_input_tensor : torch.Tensor, size [
            len_learn_window,
            num_ts_input * (var_order + 1)]
        Entire batch of all time-lag input data
            vectors for deep autoencoder serving
            as nonlinear dynamic factor model

    dae_output_tensor : torch.Tensor, size [
            len_learn_window, num_ts_output]
        Entire batch of (zero time-lag only) final
            output data vectors of deep autoencoder

    dae_error_tensor : torch.Tensor, size [
            len_learn_window, num_ts_output]
        Entire batch of error data vectors of deep
            autoencoder

**"dae_latent_code"**  

Data access method and data item(s):  

    dae_latent_code = (
        dAeDFM_obj.get_dae_latent_code())

    dae_latent_code : torch.Tensor, size [
            len_learn_window,
            num_ts_latent * (var_order + 1)]
        Latent representation code vector encoded
            at bottlenick layer of deep autoencoder
            for nonlinear dynamic factor model

**"dae_training_metric"**  

Data access method and data item(s):  

    (dae_training_loss, dae_validation_loss,
        dae_testing_loss, dae_training_score,
        dae_validation_score, dae_testing_score
    ) = dAeDFM_obj.get_dae_training_metric()

    dae_training_loss : torch.Tensor, size [
            (max_forecast_step + 1), num_ts_input]
        Training loss vector and in-sample forecasting
            loss vectors of deep autoencoder measured
            by specified loss criterion

    dae_validation_loss : torch.Tensor,
            size [num_ts_input]
        Validation loss vector of deep autoencoder
            model validation measured by specified
            loss criterion

    dae_testing_loss : torch.Tensor,
            size [num_ts_input]
        Final testing loss vector of deep auto-
            encoder final model testing measured
            by specified loss criterion

    dae_training_score : torch.Tensor, size [
            (max_forecast_step + 1), num_ts_input]
        Training score vector and in-sample forecasting
            score vectors of deep autoencoder measured
            by orrelation coefficients

    dae_validation_score : torch.Tensor,
            size [num_ts_input]
        Validation score vector of deep autoencoder
            model validation measured by correlation
            coefficients

    dae_testing_score : torch.Tensor,
            size [num_ts_input]
        Final testing score vector of deep auto-
            encoder final model testing measured
            by orrelation coefficients

**"basecase_train_metric"**  

Data access method and data item(s):  

    (basecase_train_loss, basecase_valid_loss,
        basecase_test_loss, basecase_train_score,
        basecase_valid_score, basecase_test_score
    ) = dAeDFM_obj.get_basecase_train_metric()

    basecase_train_loss : pd.DataFrame,
            index (ts_list),
            columns (0, forecast_step_list)
        Training loss and in-sample forecasting loss
            based on base-case time-series generated
            by deep autoencoder and measured by
            specified loss criterion

    basecase_valid_loss : pd.Series, index (ts_list)
        Validation loss based on base-case time-series
            generated by deep autoencoder and measured
            by specified loss criterion

    basecase_test_loss : pd.Series, index (ts_list)
        Final testing loss based on base-case time-
            series generated by deep autoencoder
            and measured by specified loss criterion

    basecase_train_score : pd.DataFrame,
            index (ts_list),
            columns (0, forecast_step_list)
        Training score and in-sample forecasting score
            based on base-case time-series generated
            by deep autoencoder and measured by
            orrelation coefficients

    basecase_valid_score : pd.Series, index (ts_list)
        Validation score based on base-case time-series
            generated by deep autoencoder and measured
            by orrelation coefficients

    basecase_test_score : pd.Series, index (ts_list)
        Final testing score based on base-case time-
            series generated by deep autoencoder and
            measured by orrelation coefficients

**"vts"**  

Data access method and data item(s):  

    vts_input, vts_output, vts_error = (
        dAeDFM_obj.get_vts())

    vts_input : pandas.DataFrame,
            index (ts_list), columns (asof_list)
        Exact copy of VTS: observed vector time-series,
            as input data for dAeDFM model learning
            and inference

    vts_output : pd.DataFrame,
            index (ts_list), columns (asof_list)
        Linear transform of output of deep autoencoder
            serving as modeled components of original
            time-series of observed vector time-series

    vts_error : pd.DataFrame,
            index (ts_list), columns (asof_list)
        Part of data values not modeled by deep auto-
            encoder, or residual component of observed
            vector time-series

**"dae_forecast_tensor"**  

Data access method and data item(s):  

    dae_forecast_tensor = (
        dAeDFM_obj.get_dae_forecast_tensor())

    dae_forecast_tensor : torch.Tensor, size [
            (max_forecast_step + 1), num_ts_output]
        Out-of-sample multi-step forecasts of zero-lag
            segment of final output of deep autoencoder

**"basecase_forecast"**  

Data access method and data item(s):  

    vts_basecase_forecast = (
        dAeDFM_obj.get_basecase_forecast())

    vts_basecase_forecast : pd.DataFrame, index (ts_list)
            columns (forecast_step_list + ['asof'])
        Out-of-sample multi-step forecasts of base-case
            time-series of observed vector time-series

**"forecast"**  

Data access method and data item(s):  

    vts_forecast = dAeDFM_obj.get_forecast()

    vts_forecast : pd.DataFrame, index (ts_list)
            columns (forecast_step_list + ['asof'])
        Out-of-sample multi-step forecasts of
            target_type time-series of observed
            vector time-series, with additional
            specification, fwd_cumsum

**"rolling_test"**  

Data access method and data item(s):  

    vts_forecast_que = MTSfUtil.rolling_TSF(
        class_name='dAeDFM',
        object_name=dAeDFM_obj,
        VTS=vector_timeseries,
        num_forecasts=num_forecasts)

    forec_score_dict = MTSfUtil.TSF_evaluate(
        VTS=vector_timeseries,
        vts_forecast_que=vts_forecast_que,
        target_type=target_type,
        fwd_cumsum=fwd_cumsum,
        half_life_list=[None],
        eval_metric_list = [
            'projection_coefficient',
            'correlation_coefficient',
            'r2_score',
            'explained_variance_score',
            'mean_absolute_percentage_error'])

    vts_forecast_que : dict, keys (forecast_asof_list)
            vts_forecast_que[asof] : DataFrame, index (ts_list),
                columns (1, 2, ..., max_forecast_step)
        Rolling out-of-sample s-step forecast of observed
            vector time-series

    forec_score_dict : dict, keys (
                (half_life, eval_metric)_1,
                (half_life, eval_metric)_2, 
                ..., 'asof')
            obj[key] = forec_score_table
        dict-Type collection of goodness scores of
            vector time-series forecasts measured
            by various evaluation parameter pairs
            (half_life, eval_metric)

Notes:

    projection_coefficient
        hyperspace projection coefficient between forecasted
        and realized time-series

    correlation_coefficient
        correlation coefficient between forecasted and
        realized time-series

Specify trained model reporting hyperparameters

In [ ]:
# available trained model reports
model_report_list = list(dict({
     1: 'mean',
     2: 'stdev',
     3: 'vts_basecase',
     4: 'dae_data_tensor',
     5: 'dae_latent_code',
     6: 'dae_training_metric',
     7: 'basecase_train_metric',
     8: 'vts',
     9: 'dae_forecast_tensor',
    10: 'basecase_forecast',
    # 11: 'forecast',
    # 12: 'rolling_test'
}).values())

# model_report_type = 'text/csv' | 'application/json'
if True:
    model_report_type = str('text/csv')

# model_report_type = 'text/csv' | 'application/json'
if False:
    model_report_type = str('application/json')

# trained model reporting hyperparameters
# all individual elements must be individual strings
my_model_report_hyperparameter = dict({
    'model_report_list': "{}".format(model_report_list),
    'model_report_type': model_report_type
})

Create model reporting job

In [ ]:
# model trainer for trained model reporting
my_trained_model_reporter = ModelTrainer(
    sagemaker_session = my_session,
    role=my_role,
    compute=my_compute,
    algorithm_name=my_algorithm_name,
    output_data_config=my_model_reporting_output_path,
    input_data_config=[  # LIST of trained model input data objects
        my_trained_model_artifact,
        my_model_reporting_input_data], 
    training_input_mode='File',
    hyperparameters=my_model_report_hyperparameter
)

Run model reporting job

In [ ]:
# prompt
print()
print('Trained model reporting\n')
print('WAIT until reporting job completes and shows trained model report files')

# ModelTrainer.train(wait=True, logs=True)
my_trained_model_reporter.train(wait=True, logs=True)

# reporting job name and trained model report path
# s3://{my_bucket}/{my_prefix}/output/data/{reporting-job-name}/output/output/
reporting_job_name = my_trained_model_reporter._latest_training_job.training_job_name
model_report_path = 's3://{}/{}/output/data/{}/output/output/'.format(my_bucket, my_prefix, reporting_job_name)
s3_bucket = my_s3_resource.Bucket(my_bucket)
model_report_folder = '{}/output/data/{}/output/output/'.format(my_prefix, reporting_job_name)

# show reporting job name and trained model report path
print('Reporting job name:\n{}\n'.format(reporting_job_name))
print('Trained model report path:\n{}\n'.format(model_report_path))
print('Trained model report files:')

# show trained model reports
for obj in s3_bucket.objects.filter(Prefix=model_report_folder):
    print(obj.key)
print()

You can review model reporting logs in AWS CloudWatch by following steps:
1. Click / Open [CloudWatch] under (AWS) "Console Home".
1. Click [Log Management] under "Logs".
1. Click [/aws/sagemaker/TrainingJobs] under "Log group".
1. Sort "Last event time" form latest to earliest.
1. Click / Open the first item under "Log stream".
1. Review the log lines of this model training job

### 4.4. Review trained model report

You can review the trained model reports as follows, for example:
1. download .CSV files to your local drive, and
1. use Microsoft Excel to review the trained model reports

### 4.5. Make rolling model test

Since dAeDFM algorithm is a nonlinear forcasting model on large vector (i.e. large set of multiple individual) time-series, a rolling model test by evaluating performances of rolling window forecasts is no doubt among the ideal approaches to test effectiveness of the model.

Specify rolling model testing hyperparameters

In [ ]:
# model_report_type = 'text/csv' | 'application/json'
if True:
    model_report_type = str('text/csv')

# model_report_type = 'text/csv' | 'application/json'
if False:
    model_report_type = str('application/json')

# rolling model (forecast) testing hyperparameters
# all individual elements must be individual strings
my_rolling_test_hyperparameter = dict({
    'model_report_list': ['rolling_test'],
    'model_report_type': model_report_type,
    'num_forecasts': '13'
})

Create rolling model testing job

In [ ]:
# model trainer for rolling model testing
my_rolling_test_handler = ModelTrainer(
    sagemaker_session = my_session,
    role=my_role,
    compute=my_compute,
    algorithm_name=my_algorithm_name,
    output_data_config=my_model_reporting_output_path,
    input_data_config=[  # LIST of trained model input data objects
        my_trained_model_artifact,
        my_model_reporting_input_data], 
    training_input_mode='File',
    hyperparameters=my_rolling_test_hyperparameter
)

Run rolling model testing job

**Notes**: You can view rolling model testing logs in CloudWatch (Console) with following steps:
1. Click / Open [CloudWatch] under (AWS) "Console Home".
1. Click [Log Management] under "Logs".
1. Check/Select [/aws/sagemaker/TrainingJobs] under "Log group".
1. Click "Start tailing" to Live Tail selected Log Group(s).

In [ ]:
# prompt
print()
print('Rolling model (forecast) testing\n')
print('WAIT until testing job completes and shows rolling test evaluation data files')

# ModelTrainer.train(wait=True, logs=True)
my_rolling_test_handler.train(wait=True, logs=True)

# testing job name and rolling test evaluation path
# s3://{my_bucket}/{my_prefix}/output/data/{training-job-name}/output/output/
testing_job_name = my_rolling_test_handler._latest_training_job.training_job_name
test_evaluation_path = 's3://{}/{}/output/data/{}/output/output/'.format(my_bucket, my_prefix, testing_job_name)
s3_bucket = my_s3_resource.Bucket(my_bucket)
test_evaluation_folder = '{}/output/data/{}/output/output/'.format(my_prefix, testing_job_name)

# show testing job name rolling test evaluation path
print('Testing job name:\n{}\n'.format(testing_job_name))
print('Rolling test evaluatoin path:\n{}\n'.format(test_evaluation_path))
print('Rolling test evaluation files:')

# show rolling test evaluation data files
for obj in s3_bucket.objects.filter(Prefix=test_evaluation_folder):
    print(obj.key)
print()

You can review rolling model testing logs live in AWS CloudWatch by following steps:
1. Click / Open [CloudWatch] under (AWS) "Console Home".
1. Click [Log Management] under "Logs".
1. Click [/aws/sagemaker/TrainingJobs] under "Log group".
1. Sort "Last event time" form latest to earliest.
1. Click / Open the first item under "Log stream".
1. Review the log lines of this model training job

### 4.6. Review rolling model test

You can review the rolling model test evaluation reports as follows, for example:
1. download .CSV files to your local drive, and
1. use Microsoft Excel to review the trained model reports

## 5. Perform Batch inference

Specify batch inference hyperparameters with CSV output

In [ ]:
# model_report_type = 'text/csv' | 'application/json'
model_report_type = str('text/csv')

# batch inference hyperparameters
# all individual elements must be individual strings
my_batch_inference_hyperparameter = dict({
    'model_report_list': ['forecast'],
    'model_report_type': model_report_type
})

Create batch inference job with CSV output

In [ ]:
# model trainer for batch inference
my_batch_inference_caller = ModelTrainer(
    sagemaker_session = my_session,
    role=my_role,
    compute=my_compute,
    algorithm_name=my_algorithm_name,
    output_data_config=my_model_reporting_output_path,
    input_data_config=[  # LIST of trained model input data objects
        my_trained_model_artifact,
        my_model_reporting_input_data], 
    training_input_mode='File',
    hyperparameters=my_batch_inference_hyperparameter
)

Run batch inference job with CSV output

In [ ]:
# prompt
print()
print('Batch model inference\n')
print('WAIT until inference job completes and shows batch inference output files')

# ModelTrainer.train(wait=True, logs=True)
my_batch_inference_caller.train(wait=True, logs=True)

# inference job name and inference output path
# s3://{my_bucket}/{my_prefix}/output/data/{inference-job-name}/output/output/
inference_job_name = my_batch_inference_caller._latest_training_job.training_job_name
inference_output_path = 's3://{}/{}/output/data/{}/output/output/'.format(my_bucket, my_prefix, inference_job_name)
s3_bucket = my_s3_resource.Bucket(my_bucket)
inference_output_folder = '{}/output/data/{}/output/output/'.format(my_prefix, inference_job_name)

# show inference job name and inference path
print('Inference job name:\n{}\n'.format(inference_job_name))
print('Batch inference output path:\n{}\n'.format(inference_output_path))
print('Batch inference output files:')

# show batch inference output data files
for obj in s3_bucket.objects.filter(Prefix=inference_output_folder):
    print(obj.key)
print()

Specify batch inference hyperparameters with JSON output

In [ ]:
# model_report_type = 'text/csv' | 'application/json'
model_report_type = str('application/json')

# batch inference hyperparameters
# all individual elements must be individual strings
my_batch_inference_hyperparameter = dict({
    'model_report_list': ['forecast'],
    'model_report_type': model_report_type
})

Create batch inference job with JSON output

In [ ]:
# model trainer for batch inference
my_batch_inference_caller = ModelTrainer(
    sagemaker_session = my_session,
    role=my_role,
    compute=my_compute,
    algorithm_name=my_algorithm_name,
    output_data_config=my_model_reporting_output_path,
    input_data_config=[  # LIST of trained model input data objects
        my_trained_model_artifact,
        my_model_reporting_input_data], 
    training_input_mode='File',
    hyperparameters=my_batch_inference_hyperparameter
)

Run batch inference job with JSON output

In [ ]:
# prompt
print()
print('Batch model inference\n')
print('WAIT until inference job completes and shows batch inference output files')

# ModelTrainer.train(wait=True, logs=True)
my_batch_inference_caller.train(wait=True, logs=True)

# inference job name and inference output path
# s3://{my_bucket}/{my_prefix}/output/data/{inference-job-name}/output/output/
inference_job_name = my_batch_inference_caller._latest_training_job.training_job_name
inference_output_path = 's3://{}/{}/output/data/{}/output/output/'.format(my_bucket, my_prefix, inference_job_name)
s3_bucket = my_s3_resource.Bucket(my_bucket)
inference_output_folder = '{}/output/data/{}/output/output/'.format(my_prefix, inference_job_name)

# show inference job name and inference path
print('Inference job name:\n{}\n'.format(inference_job_name))
print('Batch inference output path:\n{}\n'.format(inference_output_path))
print('Batch inference output files:')

# show batch inference output data files
for obj in s3_bucket.objects.filter(Prefix=inference_output_folder):
    print(obj.key)
print()

You can review the batch inference output as follows, for example:
1. download .CSV files to your local drive, and
1. use Microsoft Excel to review the trained model reports

## 6. Clean-up

### 6.1. Delete the model

In [ ]:
# clear model_trainer def
my_model_trainer = None
my_trained_model_reporter = None
my_rolling_test_handler = None
my_batch_inference_caller = None

### 6.2. Unsubscribe to the listing (optional)

If you would like to unsubscribe to the algorithm, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note - You can find this information by looking at the container name associated with the model. 

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__  to cancel the subscription.



## Technical support

For technical support, pleas email to: 
prod.i4cast@gmail.com